# 🛰️ Sentinel Stockpile — Exploration Notebook

Use this notebook to:
1. Visually inspect fetched imagery (RGB composites)
2. Explore spectral index distributions
3. Tune classification thresholds
4. Compare classified output against true color
5. Validate stockpile measurements

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio

sys.path.insert(0, "../src")
from preprocess import load_scene_stack, to_reflectance
from classify import compute_indices, classify_pixels, CLASS_NAMES

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 100

## 1. Pick a site and scene

In [ ]:
SITE = "longview_port"  # Change this to your site

# Load site config
with open(f"../config/sites/{SITE}.json") as f:
    site_config = json.load(f)
print(f"Site: {site_config['name']}")
print(f"Commodity: {site_config['commodity']}")

# List available scenes
imagery_dir = Path(f"../output/{SITE}/imagery")
scenes = sorted([d.name for d in imagery_dir.iterdir() if d.is_dir()])
print(f"\nAvailable scenes ({len(scenes)}):")
for s in scenes:
    print(f"  {s[:4]}-{s[4:6]}-{s[6:8]}")

In [ ]:
# Pick a scene to explore
SCENE_DATE = scenes[-1]  # Most recent; change as needed
scene_dir = imagery_dir / SCENE_DATE
print(f"Exploring scene: {SCENE_DATE}")

## 2. True Color Composite

In [ ]:
bands, profile = load_scene_stack(scene_dir)

# RGB composite (B04=Red, B03=Green, B02=Blue)
# Stretch for visualization — adjust gain if too dark/bright
GAIN = 3.0  # increase if image looks dark

rgb = np.stack([
    np.clip(to_reflectance(bands["B04"]) * GAIN, 0, 1),
    np.clip(to_reflectance(bands["B03"]) * GAIN, 0, 1),
    np.clip(to_reflectance(bands["B02"]) * GAIN, 0, 1),
], axis=-1)

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(rgb)
ax.set_title(f"True Color — {SCENE_DATE[:4]}-{SCENE_DATE[4:6]}-{SCENE_DATE[6:8]}", fontsize=14)
plt.tight_layout()
plt.show()
print(f"Image shape: {rgb.shape[1]}x{rgb.shape[0]} pixels at 10m = {rgb.shape[1]*10}m x {rgb.shape[0]*10}m")

## 3. False Color Composites

False color helps distinguish materials:
- **NIR-Red-Green**: vegetation pops bright red, water is dark, built surfaces are cyan/gray
- **SWIR-NIR-Red**: lumber and bare soil separate better, moisture differences visible

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# True color
axes[0].imshow(rgb)
axes[0].set_title("True Color (R-G-B)")

# NIR false color
nir_fc = np.stack([
    np.clip(to_reflectance(bands["B08"]) * GAIN, 0, 1),
    np.clip(to_reflectance(bands["B04"]) * GAIN, 0, 1),
    np.clip(to_reflectance(bands["B03"]) * GAIN, 0, 1),
], axis=-1)
axes[1].imshow(nir_fc)
axes[1].set_title("False Color (NIR-R-G)")

# SWIR false color
swir_fc = np.stack([
    np.clip(to_reflectance(bands["B11"]) * GAIN, 0, 1),
    np.clip(to_reflectance(bands["B08"]) * GAIN, 0, 1),
    np.clip(to_reflectance(bands["B04"]) * GAIN, 0, 1),
], axis=-1)
axes[2].imshow(swir_fc)
axes[2].set_title("False Color (SWIR-NIR-R)")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## 4. Spectral Index Maps

These are the indices we use for classification. Look at where stockpile material
falls in each index to decide if thresholds need adjustment.

In [ ]:
indices = compute_indices(bands)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

index_configs = [
    ("ndvi", "NDVI (Vegetation)", "RdYlGn", -0.5, 0.8),
    ("ndmi", "NDMI (Moisture)", "RdYlBu", -0.5, 0.5),
    ("bsi", "BSI (Bare Soil)", "RdYlBu_r", -0.4, 0.4),
    ("ndwi", "NDWI (Water)", "RdYlBu", -0.5, 0.5),
    ("brightness", "Brightness", "gray", 0, 0.3),
    ("swir_ratio", "SWIR Ratio", "coolwarm", -0.3, 0.3),
]

for ax, (idx_name, title, cmap, vmin, vmax) in zip(axes.flat, index_configs):
    im = ax.imshow(indices[idx_name], cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(f"Spectral Indices — {SCENE_DATE}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Index Histograms

Look for bimodal distributions — the gap between peaks is where your
classification threshold should sit.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, (idx_name, title, _, vmin, vmax) in zip(axes.flat, index_configs):
    data = indices[idx_name].flatten()
    data = data[np.isfinite(data)]
    ax.hist(data, bins=100, color="#2196F3", alpha=0.7, edgecolor="none")
    ax.set_title(title)
    ax.set_xlim(vmin, vmax)
    ax.axvline(x=np.median(data), color="red", linestyle="--", label=f"median={np.median(data):.3f}")
    ax.legend(fontsize=8)

plt.suptitle("Index Distributions (look for bimodal peaks)", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Classification Result

In [ ]:
commodity = site_config.get("commodity", "lumber")
classified = classify_pixels(indices, commodity)

# Color map
colors = ["#1a1a2e", "#1e81b0", "#4a7c59", "#d4a843", "#8c7a6b"]
cmap = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(rgb)
axes[0].set_title("True Color", fontsize=13)

im = axes[1].imshow(classified, cmap=cmap, norm=norm)
axes[1].set_title("Classification", fontsize=13)

patches = [plt.Rectangle((0,0),1,1, fc=colors[i]) for i in [1,2,3,4]]
labels = ["Water", "Vegetation", "Stockpile", "Ground"]
axes[1].legend(patches, labels, loc="upper right", fontsize=10)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(f"Classification Validation — {SCENE_DATE}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Stats
unique, counts = np.unique(classified, return_counts=True)
total = counts.sum()
print("\nPixel counts:")
for u, c in zip(unique, counts):
    name = CLASS_NAMES.get(u, "?")
    pct = c / total * 100
    area_m2 = c * 100  # 10m x 10m pixels
    print(f"  {name:12s}: {c:>6d} pixels ({pct:5.1f}%) = {area_m2:>10,.0f} m²")

## 7. Threshold Tuning

Adjust thresholds below and re-run to see the effect.
The goal: stockpile pixels (amber) should cover actual lumber/container stacks,
not bleed into pavement or vegetation.

In [ ]:
from classify import THRESHOLDS
import copy

# Copy and modify thresholds
custom = copy.deepcopy(THRESHOLDS[commodity])

# --- ADJUST THESE ---
# custom["veg_ndvi"] = 0.30          # lower = more vegetation classified
# custom["stockpile_brightness_min"] = 0.06  # lower = catches darker material
# custom["stockpile_brightness_max"] = 0.40  # higher = catches brighter material
# custom["stockpile_bsi_min"] = -0.15        # lower = more permissive
# --------------------

print(f"Current thresholds for '{commodity}':")
for k, v in custom.items():
    print(f"  {k}: {v}")

# Re-classify with custom thresholds (manual override)
t = custom
reclassified = np.full(indices["ndvi"].shape, 4, dtype=np.uint8)
water = indices["ndwi"] > t["water_ndwi"]
veg = (~water) & (indices["ndvi"] > t["veg_ndvi"])
stockpile = (~water) & (~veg) & \
    (indices["bsi"] > t["stockpile_bsi_min"]) & \
    (indices["brightness"] > t["stockpile_brightness_min"]) & \
    (indices["brightness"] < t["stockpile_brightness_max"])
reclassified[water] = 1
reclassified[veg] = 2
reclassified[stockpile] = 3

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
axes[0].imshow(rgb)
axes[0].set_title("True Color")
axes[1].imshow(classified, cmap=cmap, norm=norm)
axes[1].set_title("Original Thresholds")
axes[2].imshow(reclassified, cmap=cmap, norm=norm)
axes[2].set_title("Custom Thresholds")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## 8. Time Series Validation

Load the measured time series and check for anomalies.

In [ ]:
ts_path = Path(f"../output/{SITE}/time_series.json")
if ts_path.exists():
    with open(ts_path) as f:
        ts = json.load(f)
    
    dates = [e["date"] for e in ts]
    areas = [e["stockpile_m2"] for e in ts]
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(dates, areas, "o-", color="#8b6914", linewidth=2)
    ax.fill_between(range(len(dates)), areas, alpha=0.2, color="#d4a843")
    ax.set_xticks(range(len(dates)))
    ax.set_xticklabels(dates, rotation=45, ha="right")
    ax.set_ylabel("Stockpile Area (m²)")
    ax.set_title(f"Stockpile Time Series — {site_config['name']}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Flag large swings (>30% change) as possible cloud contamination
    print("\nLarge changes (possible cloud/shadow artifacts):")
    for i in range(1, len(areas)):
        if areas[i-1] > 0:
            pct_change = (areas[i] - areas[i-1]) / areas[i-1] * 100
            if abs(pct_change) > 30:
                print(f"  ⚠ {dates[i]}: {pct_change:+.1f}% — inspect this scene visually")
else:
    print("No time series found. Run the full pipeline first.")

## 9. Pixel Sampling Tool

Click-equivalent: sample spectral values at known locations to understand
what lumber vs. pavement vs. water looks like spectrally.

In [ ]:
# Sample specific pixel locations
# Set these by looking at the RGB composite above
sample_points = {
    "lumber_stack": (30, 45),    # (row, col) — adjust to match your image
    "pavement": (50, 50),
    "water": (10, 10),
    "vegetation": (70, 70),
}

print(f"{'Location':20s} {'NDVI':>8s} {'NDMI':>8s} {'BSI':>8s} {'NDWI':>8s} {'Bright':>8s} {'Class':>10s}")
print("-" * 80)

for name, (row, col) in sample_points.items():
    if row < indices["ndvi"].shape[0] and col < indices["ndvi"].shape[1]:
        cls = CLASS_NAMES.get(classified[row, col], "?")
        print(f"{name:20s} "
              f"{indices['ndvi'][row,col]:>8.3f} "
              f"{indices['ndmi'][row,col]:>8.3f} "
              f"{indices['bsi'][row,col]:>8.3f} "
              f"{indices['ndwi'][row,col]:>8.3f} "
              f"{indices['brightness'][row,col]:>8.3f} "
              f"{cls:>10s}")
    else:
        print(f"{name:20s}  (out of bounds — adjust coordinates)")

---

## Next Steps

Once thresholds look good:
1. Update `THRESHOLDS` in `src/classify.py`
2. Re-run: `pixi run pipeline --site longview_port`
3. Check the time series for consistency
4. Add more sites in `config/sites/`